In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/shl-intern-hiring-assessment/Dataset/sample_submission.csv
/kaggle/input/shl-intern-hiring-assessment/Dataset/train.csv
/kaggle/input/shl-intern-hiring-assessment/Dataset/test.csv
/kaggle/input/shl-intern-hiring-assessment/Dataset/audios/test/audio_885.wav
/kaggle/input/shl-intern-hiring-assessment/Dataset/audios/test/audio_1142.wav
/kaggle/input/shl-intern-hiring-assessment/Dataset/audios/test/audio_1006.wav
/kaggle/input/shl-intern-hiring-assessment/Dataset/audios/test/audio_817.wav
/kaggle/input/shl-intern-hiring-assessment/Dataset/audios/test/audio_765.wav
/kaggle/input/shl-intern-hiring-assessment/Dataset/audios/test/audio_508.wav
/kaggle/input/shl-intern-hiring-assessment/Dataset/audios/test/audio_257.wav
/kaggle/input/shl-intern-hiring-assessment/Dataset/audios/test/audio_330.wav
/kaggle/input/shl-intern-hiring-assessment/Dataset/audios/test/audio_72.wav
/kaggle/input/shl-intern-hiring-assessment/Dataset/audios/test/audio_328.wav
/kaggle/input/shl-intern-hiring-ass

# Using OpenAI's Whisper for grammar scoring

## Brief Report

### Objective
The goal is to predict grammar scores (1 to 5) for spoken English audio samples using a machine learning pipeline. The evaluation metric is Pearson correlation.

### Approach Overview
1. **Transcription**: Use OpenAI's Whisper model to transcribe the audio.
2. **Feature Engineering**: Extract handcrafted linguistic features from the transcribed text.
3. **Regression Model**: Use a Gradient Boosting Regressor to predict grammar scores based on features.
 
### Preprocessing Steps
- Audio is loaded using `librosa` at 16kHz sampling rate.
- Whisper is used for transcription in English.
- Extracted features include word count, unique word ratio, grammatical constructs, etc.
 
### Model & Pipeline Architecture
- Whisper for transcription.
- Feature engineering from transcripts.
- Gradient Boosting Regressor (200 estimators, learning rate 0.05).

### Evaluation
- Validation set evaluated using Pearson correlation.


## Importing libraries and initial set up

In [2]:
# ─── Install dependencies ────────────────────────────────────────────────────
!pip install -q transformers librosa pandas scikit-learn torch tqdm xgboost language-tool-python spacy textstat
!python -m spacy download en_core_web_sm -q
!sudo apt install -y ffmpeg -q

import os, warnings, torch, librosa, numpy as np, pandas as pd
from tqdm.auto import tqdm
from scipy.stats import pearsonr
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from xgboost import XGBRegressor
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import spacy, textstat, language_tool_python

warnings.filterwarnings("ignore")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.1/177.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 37.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 16.0 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 129 not upgraded.
Using device: cuda


## Loading whisper for transcription

In [3]:
# Adoptium mirrors are more reliable than Ubuntu's universe repo
!wget -q https://github.com/adoptium/temurin17-binaries/releases/download/jdk-17.0.10%2B7/OpenJDK17U-jdk_x64_linux_hotspot_17.0.10_7.tar.gz
!tar -xzf OpenJDK17U-jdk_x64_linux_hotspot_17.0.10_7.tar.gz -C /usr/local/
import os
os.environ["JAVA_HOME"] = "/usr/local/jdk-17.0.10+7"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]
!java -version  # should now show 17

openjdk version "17.0.10" 2024-01-16
OpenJDK Runtime Environment Temurin-17.0.10+7 (build 17.0.10+7)
OpenJDK 64-Bit Server VM Temurin-17.0.10+7 (build 17.0.10+7, mixed mode, sharing)


In [4]:
# ─── Load models (do this ONCE outside the feature loop) ─────────────────────

# Use medium for better transcription quality than small
processor = WhisperProcessor.from_pretrained("openai/whisper-large-v3")
whisper   = WhisperForConditionalGeneration.from_pretrained("openai/whisper-large-v3").to(device)
forced_ids = processor.get_decoder_prompt_ids(language="en", task="transcribe")

nlp  = spacy.load("en_core_web_sm")
tool = language_tool_python.LanguageTool("en-US")

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

## Feature extraction function

 | Feature | Description |
 |--------|-------------|
 | `len(words)` | Total number of words – indicates fluency or verbosity |
 | `len(set(words)) / max(1, len(words))` | Lexical diversity – how many unique words are used |
 | `transcript.count('.')` | Approximate number of sentences – gives a hint of structure |
 | `sum(w.endswith('ing') for w in words)` | Count of words ending with '-ing' – suggests continuous tenses or gerunds |
 | `sum(w.endswith('ed') for w in words)` | Count of words ending with '-ed' – helps identify past tense verbs |
 | `sum(len(w) > 6 for w in words)` | Number of long words (length > 6) – may reflect vocabulary complexity |
 | `transcript.count('the')` | Frequency of the word 'the' – related to proper article usage |

In [5]:
# ─── Feature extraction ───────────────────────────────────────────────────────

def get_whisper_features(audio_path: str):
    audio, sr = librosa.load(audio_path, sr=16000)
    inputs = processor(audio, sampling_rate=sr, return_tensors="pt").to(device)

    with torch.no_grad():
        enc_out = whisper.model.encoder(
            inputs.input_features,
            attention_mask=inputs.get("attention_mask", None)
        )
        embeds = enc_out.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()

        ids = whisper.generate(
            **inputs,
            forced_decoder_ids=forced_ids,
            attention_mask=inputs.get("attention_mask",
                           torch.ones_like(inputs.input_features[:, 0, :]).long())
        )
        transcript = processor.decode(ids[0], skip_special_tokens=True)

    return transcript, embeds


def get_linguistic_features(transcript: str) -> np.ndarray:
    """
    ~25 hand-crafted grammatical / complexity features from the transcript.
    These capture things the audio embedding misses (explicit grammar errors,
    clause depth, tense variety, vocabulary breadth, etc.).
    """
    if not transcript.strip():
        return np.zeros(25)

    doc   = nlp(transcript)
    words = [t.text.lower() for t in doc if t.is_alpha]
    sents = list(doc.sents)

    # --- Basic counts ---
    n_words  = max(len(words), 1)
    n_sents  = max(len(sents), 1)
    n_unique = len(set(words))

    # --- Vocabulary richness ---
    type_token_ratio = n_unique / n_words
    avg_word_len     = np.mean([len(w) for w in words]) if words else 0
    long_words_ratio = sum(len(w) > 6 for w in words) / n_words

    # --- Sentence structure ---
    sent_lengths = [len([t for t in s if t.is_alpha]) for s in sents]
    avg_sent_len  = np.mean(sent_lengths)
    std_sent_len  = np.std(sent_lengths)

    # --- POS diversity (good grammar uses varied parts of speech) ---
    pos_counts = {}
    for t in doc:
        pos_counts[t.pos_] = pos_counts.get(t.pos_, 0) + 1
    total_pos   = sum(pos_counts.values()) or 1
    pos_entropy = -sum((c/total_pos)*np.log2(c/total_pos) for c in pos_counts.values())
    subj_ratio  = pos_counts.get("NOUN", 0) / n_words
    verb_ratio  = pos_counts.get("VERB", 0) / n_words
    adj_ratio   = pos_counts.get("ADJ",  0) / n_words
    adv_ratio   = pos_counts.get("ADV",  0) / n_words
    aux_ratio   = pos_counts.get("AUX",  0) / n_words

    # --- Dependency depth (proxy for syntactic complexity) ---
    def dep_depth(token):
        d = 0
        while token.head != token:
            token = token.head
            d += 1
        return d
    depths    = [dep_depth(t) for t in doc]
    avg_depth = np.mean(depths) if depths else 0
    max_depth = np.max(depths)  if depths else 0

    # --- Tense variety ---
    tenses   = {t.morph.get("Tense", [""])[0] for t in doc if t.pos_ == "VERB"}
    n_tenses = len(tenses - {""})

    # --- Readability / complexity scores ---
    fk_grade  = textstat.flesch_kincaid_grade(transcript)
    flesch    = textstat.flesch_reading_ease(transcript)

    # --- Grammar errors (LanguageTool) ---
    matches = tool.check(transcript)
    gram_errs = 0
    for m in matches:
    # Try ruleIssueType first (LT latest), fall back to category (LT 6.3)
        issue_type = getattr(m, "ruleIssueType",
                 getattr(m, "category", "")).lower()
        if issue_type in ("grammar", "style", "typographical", "punctuation"):
            gram_errs += 1
    err_rate = gram_errs / n_words

    # --- Subordinate clause proxies ---
    subord_markers = {"although","because","since","while","if","unless",
                      "when","after","before","as","that","which","who"}
    sub_ratio = sum(t.text.lower() in subord_markers for t in doc) / n_words

    return np.array([
        type_token_ratio, avg_word_len, long_words_ratio,
        avg_sent_len, std_sent_len,
        pos_entropy, subj_ratio, verb_ratio, adj_ratio, adv_ratio, aux_ratio,
        avg_depth, max_depth,
        n_tenses,
        fk_grade, flesch,
        gram_errs, err_rate,
        sub_ratio,
        n_words, n_sents, n_unique,
        len(transcript),
        pos_counts.get("PROPN", 0) / n_words,  # proper nouns (name fluency)
        pos_counts.get("CCONJ", 0) / n_words,  # coordinating conjunctions
    ])


def extract_all_features(audio_path: str):
    transcript, embeds    = get_whisper_features(audio_path)
    ling_feats            = get_linguistic_features(transcript)
    return np.concatenate([embeds, ling_feats])   # (768 + 25,) = (793,)


## Loading dataset and training it

In [6]:
# ─── Load data ────────────────────────────────────────────────────────────────

TRAIN_CSV   = "/kaggle/input/shl-intern-hiring-assessment/Dataset/train.csv"
TEST_CSV    = "/kaggle/input/shl-intern-hiring-assessment/Dataset/test.csv"
TRAIN_AUDIO = "/kaggle/input/shl-intern-hiring-assessment/Dataset/audios/train"
TEST_AUDIO  = "/kaggle/input/shl-intern-hiring-assessment/Dataset/audios/test"

train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

In [7]:
# ─── Extract features (this is the slow part – ~2–3 min on GPU) ──────────────

print("Extracting training features...")
X_train = np.array([extract_all_features(f"{TRAIN_AUDIO}/{f}")
                    for f in tqdm(train_df["filename"])])
y_train = train_df["label"].values

print("Extracting test features...")
X_test = np.array([extract_all_features(f"{TEST_AUDIO}/{f}")
                   for f in tqdm(test_df["filename"])])

Extracting training features...


  0%|          | 0/444 [00:00<?, ?it/s]

Extracting test features...


  0%|          | 0/204 [00:00<?, ?it/s]

In [8]:
# ─── Scale ───────────────────────────────────────────────────────────────────

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

In [9]:
# ─── Models ───────────────────────────────────────────────────────────────────

ridge = Ridge(alpha=10.0)
xgb   = XGBRegressor(
    n_estimators   = 400,
    learning_rate  = 0.03,
    max_depth      = 4,
    subsample      = 0.8,
    colsample_bytree = 0.6,
    random_state   = 42,
    tree_method    = "hist",
    device         = "cuda" if torch.cuda.is_available() else "cpu",
)

# Cross-validated Pearson on training set
for name, model in [("Ridge", ridge), ("XGBoost", xgb)]:
    scores = cross_val_score(model, X_train, y_train,
                             cv=5, scoring="r2")
    print(f"{name} CV R² (proxy): {scores.mean():.3f} ± {scores.std():.3f}")

# Fit on full training set
ridge.fit(X_train, y_train)
xgb.fit(X_train, y_train)


Ridge CV R² (proxy): -4.102 ± 3.186
XGBoost CV R² (proxy): -3.282 ± 3.539


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.6, device='cuda', early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.03, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=4, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=400, n_jobs=None,
             num_parallel_tree=None, random_state=42, ...)

In [10]:
# ─── Ensemble prediction ──────────────────────────────────────────────────────

# Equal blend — you can tune weights via a held-out val split
pred_ridge = ridge.predict(X_test)
pred_xgb   = xgb.predict(X_test)
pred_final = np.clip(0.4*pred_ridge + 0.6*pred_xgb, 1, 5)


In [11]:
# ─── Submission ───────────────────────────────────────────────────────────────

submission = pd.DataFrame({"filename": test_df["filename"], "label": pred_final})
submission.to_csv("submission.csv", index=False)
print(submission.describe())
print(submission.head(10))

            label
count  204.000000
mean     3.708799
std      0.777230
min      1.956275
25%      3.099005
50%      3.757687
75%      4.330168
max      5.000000
         filename     label
0   audio_804.wav  2.759044
1  audio_1028.wav  2.223446
2   audio_865.wav  2.342687
3   audio_774.wav  2.616616
4  audio_1138.wav  2.102272
5   audio_278.wav  3.115216
6  audio_1212.wav  3.216227
7   audio_178.wav  3.554166
8   audio_542.wav  3.053824
9   audio_248.wav  2.782280


In [12]:
from scipy.stats import pearsonr
from sklearn.model_selection import train_test_split

# ─── Hold-out validation (do this BEFORE fitting on full data) ────────────────

X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

# Fit on 80%
ridge.fit(X_tr, y_tr)
xgb.fit(X_tr, y_tr)

# Predict on 20%
pred_ridge_val = ridge.predict(X_val)
pred_xgb_val   = xgb.predict(X_val)
pred_blend_val = np.clip(0.4*pred_ridge_val + 0.6*pred_xgb_val, 1, 5)

# Pearson for each
r_ridge, _ = pearsonr(y_val, pred_ridge_val)
r_xgb,   _ = pearsonr(y_val, pred_xgb_val)
r_blend, _ = pearsonr(y_val, pred_blend_val)

print(f"Ridge   Pearson: {r_ridge:.4f}")
print(f"XGBoost Pearson: {r_xgb:.4f}")
print(f"Blend   Pearson: {r_blend:.4f}")

# ─── Now refit on FULL training data for submission ───────────────────────────
ridge.fit(X_train, y_train)
xgb.fit(X_train, y_train)

Ridge   Pearson: 0.6838
XGBoost Pearson: 0.8039
Blend   Pearson: 0.7746


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.6, device='cuda', early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.03, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=4, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=400, n_jobs=None,
             num_parallel_tree=None, random_state=42, ...)